# Tiền xử lý dữ liệu bệnh & triệu chứng

Notebook này **chỉ xử lý 2 file dữ liệu** trong thư mục `01_data`:
- `benh_trieuchung.csv` — mỗi dòng là một ca bệnh với các triệu chứng.
- `trongso_mucdo_nghiemtrong.csv` — trọng số (mức độ nặng) của từng triệu chứng.

**Mục tiêu:** biến dữ liệu chữ thành **bảng số** để máy học được, rồi lưu ra `features.csv`.

> Việc xử lý câu mô tả của bệnh nhân (NLP) nằm ở notebook riêng: `xu_ly_input_nlp.ipynb`.


## Bước 1. Đọc 2 file dữ liệu

In [1]:
import pandas as pd

benh = pd.read_csv("../01_data/benh_trieuchung.csv")
print("Số ca bệnh:", len(benh), "| Số bệnh khác nhau:", benh["Benh"].nunique())
benh.head()


Số ca bệnh: 5000 | Số bệnh khác nhau: 400


,Benh,TrieuChung_1,TrieuChung_2,TrieuChung_3,TrieuChung_4,TrieuChung_5,TrieuChung_6,TrieuChung_7,TrieuChung_8
0,Viêm bờ mi mạn,mắt đỏ,mờ mắt,đau mắt,sốt,NaN,NaN,NaN,NaN
1,Quai bị,sưng tuyến mang tai,sốt,đau khi nhai,mệt mỏi,NaN,NaN,NaN,NaN
2,Viêm thận cấp,tiểu rắt,tiểu nhiều lần,tiểu buốt,đau lưng,tiểu ra máu,NaN,NaN,NaN
3,Viêm van tim mạn,đau ngực,mệt mỏi,sốt,khó thở,NaN,NaN,NaN,NaN
4,Viêm túi mật cấp,mệt mỏi,đau dữ dội,chán ăn,buồn nôn,vàng da,NaN,NaN,NaN


In [2]:
trongso = pd.read_csv("../01_data/trongso_mucdo_nghiemtrong.csv")
print("Số triệu chứng có trọng số:", len(trongso))
trongso.head()


Số triệu chứng có trọng số: 135


,TrieuChung,TrongSo
0,bong tróc da,2
1,buồn bã kéo dài,4
2,buồn nôn,3
3,bầm tím,3
4,chán ăn,3


## Bước 2. Tạo "từ điển" trọng số

Đổi bảng trọng số thành `dict` để tra cứu nhanh: tên triệu chứng → mức độ nặng.

In [3]:
trong_so = dict(zip(trongso["TrieuChung"], trongso["TrongSo"]))
ds_trieuchung = list(trongso["TrieuChung"])   # danh sách triệu chứng (thứ tự cột)
print("Ví dụ:", {k: trong_so[k] for k in ds_trieuchung[:5]})


Ví dụ: {'bong tróc da': 2, 'buồn bã kéo dài': 4, 'buồn nôn': 3, 'bầm tím': 3, 'chán ăn': 3}


## Bước 3. Mã hóa one-hot có trọng số

Mỗi **triệu chứng là một cột**. Với mỗi ca bệnh:
- Nếu **có** triệu chứng đó → điền **trọng số** của nó.
- Nếu **không có** → điền 0.

Cột `Benh` là nhãn cần dự đoán.


In [4]:
cot_trieuchung = [c for c in benh.columns if c.startswith("TrieuChung")]

X = pd.DataFrame(0, index=benh.index, columns=ds_trieuchung)
for i in range(len(benh)):
    for cot in cot_trieuchung:
        tc = benh.loc[i, cot]
        if isinstance(tc, str) and tc.strip() in trong_so:
            X.loc[i, tc.strip()] = trong_so[tc.strip()]

X["Benh"] = benh["Benh"]
print("Bảng đặc trưng:", X.shape)
X.head()


Bảng đặc trưng: (5000, 136)


,bong tróc da,buồn bã kéo dài,buồn nôn,bầm tím,chán ăn,chóng mặt,chảy máu cam,chảy mủ tai,chảy nước mắt,chậm chạp,...,đói nhiều,đầy hơi,đỏ khớp,đổ mồ hôi,đổ mồ hôi lạnh,đổ mồ hôi đêm,ớn lạnh,ợ chua,ợ nóng,Benh
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Viêm bờ mi mạn
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Quai bị
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Viêm thận cấp
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Viêm van tim mạn
4,0,0,3,0,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Viêm túi mật cấp


## Bước 4. Lưu kết quả ra file

In [5]:
import os
os.makedirs("../01_data/processed", exist_ok=True)
X.to_csv("../01_data/processed/features.csv", index=False, encoding="utf-8-sig")
print("Đã lưu: 01_data/processed/features.csv")


Đã lưu: 01_data/processed/features.csv
